Customer Support Router

In [1]:
pip install -U langchain


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.4/245.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 8.8 MB/s eta 0:00:00
  Attempting uninstall: langgraph-sdk
    Found existing installation: langgraph-sdk 0.3.14
    Uninstalling langgraph-sdk-0.3.14:
      Successfully uninstalled langgraph-sdk-0.3.14
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.2.1
    Uninstalling langgraph-1.2.1:
      Successfully uninstalled langgraph-1.2.1
  Attempting uninstall: langchain
    Found existing installation: langchain 1.3.1
    Uninstalling langchain-1.3.1:
      Successfully uninstalled langchain-1.3.1


In [2]:
pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 1.0 MB/s eta 0:00:00


In [14]:
from typing import TypedDict
from langgraph.graph import StateGraph, END
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# LLM
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=userdata.get("apikey")
)


# State
class SupportState(TypedDict):
    query: str
    category: str
    response: str

In [5]:
def classifier_node(state: SupportState):
    prompt = f"""
Classify the customer query into one category only:

billing
technical
complaint

Query:
{state['query']}

Return only the category name.
"""

    result = llm.invoke([HumanMessage(content=prompt)])

    return {
        "category": result.content.strip().lower()
    }


In [6]:
def billing_agent(state: SupportState):

    response = f"""
Billing Support:

Customer Query:
{state['query']}

Possible issues:
- Payment failed
- Refund status
- Subscription charges
- Invoice requests
"""

    return {"response": response}


In [7]:
def technical_agent(state: SupportState):

    response = f"""
Technical Support:

Customer Query:
{state['query']}

Possible solutions:
- Restart application
- Clear cache
- Check internet connection
- Reinstall software
"""

    return {"response": response}


In [8]:
def complaint_agent(state: SupportState):

    response = f"""
Complaint Support:

Customer Query:
{state['query']}

We apologize for the inconvenience.
Your complaint has been recorded and escalated.
"""

    return {"response": response}

In [9]:
def route_query(state: SupportState):

    category = state["category"]

    if "billing" in category:
        return "billing"

    elif "technical" in category:
        return "technical"

    else:
        return "complaint"

In [10]:
workflow = StateGraph(SupportState)

workflow.add_node("classifier", classifier_node)
workflow.add_node("billing", billing_agent)
workflow.add_node("technical", technical_agent)
workflow.add_node("complaint", complaint_agent)

workflow.set_entry_point("classifier")

workflow.add_conditional_edges(
    "classifier",
    route_query,
    {
        "billing": "billing",
        "technical": "technical",
        "complaint": "complaint"
    }
)

workflow.add_edge("billing", END)
workflow.add_edge("technical", END)
workflow.add_edge("complaint", END)

app = workflow.compile()

In [16]:
query = input("Enter Customer Query: ")

result = app.invoke({
    "query": query,
    "category": "",
    "response": ""
})

print("\nCategory:", result["category"])
print("\nFinal Response:")
print(result["response"])

Enter Customer Query: my billing is done twice

Category: billing

Final Response:

Billing Support:

Customer Query:
my billing is done twice

Possible issues:
- Payment failed
- Refund status
- Subscription charges
- Invoice requests

